In [1]:
# Cell 1 - Project setup

import os
import sys
import json
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import skrf as rf

PROJECT_ROOT = "/home/tekb/Master_Thesis_KB/CodesAndData"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)

PROJECT_ROOT: /home/tekb/Master_Thesis_KB/CodesAndData
Device: cuda


In [2]:
# Cell 2 - EM validation configuration for nearest-neighbor inverse design

EM_CFG = {
    # Input folder containing EM .s4p files
    "em_input_dir": os.path.join(
        PROJECT_ROOT,
        "Data",
        "nn_inverse_design",
    ),

    # Output folder for validation results
    "em_results_dir": os.path.join(
        PROJECT_ROOT,
        "Results",
        "EM_Validation_NN",
    ),

    # Impedance cache used as reference database
    "impedance_cache": os.path.join(
        PROJECT_ROOT,
        "Data",
        "stage2_impedance_cache_ln_phase_full20k.pt",
    ),

    # Expected file pattern:
    # target_100005_start_116459_geometry_impedance_loss_out.s4p
    "filename_regex": r"target_(\d+)_start_(\d+)_geometry_impedance_loss_out\.s4p",

    # Plot settings
    "save_plots": True,
    "dpi": 300,
}

os.makedirs(EM_CFG["em_results_dir"], exist_ok=True)

print("EM input folder:", EM_CFG["em_input_dir"])
print("EM results folder:", EM_CFG["em_results_dir"])

EM input folder: /home/tekb/Master_Thesis_KB/CodesAndData/Data/nn_inverse_design
EM results folder: /home/tekb/Master_Thesis_KB/CodesAndData/Results/EM_Validation_NN


In [3]:
# Cell 3 - Load impedance cache

cache_path = EM_CFG["impedance_cache"]

if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Impedance cache not found:\n{cache_path}")

imp_cache = torch.load(cache_path, map_location="cpu")

cache_sids = imp_cache["simu_ids"].cpu().numpy().astype(int)
sid_to_cache_pos = {int(s): i for i, s in enumerate(cache_sids)}

freq_hz = np.asarray(imp_cache["freq_hz"], dtype=np.float64)
freq_mhz = freq_hz / 1e6

impedance_data = imp_cache["impedance"]

print("Loaded impedance cache:")
print(cache_path)
print("Impedance shape:", impedance_data.shape)
print("Frequency points:", len(freq_hz))
print("Number of SIDs:", len(cache_sids))

Loaded impedance cache:
/home/tekb/Master_Thesis_KB/CodesAndData/Data/stage2_impedance_cache_ln_phase_full20k.pt
Impedance shape: torch.Size([20000, 2, 334])
Frequency points: 334
Number of SIDs: 20000


/tmp/ipykernel_325892/2131589225.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  imp_cache = torch.load(cache_path, map_location="cpu")


In [4]:
# Cell 4 - Helper functions

def get_true_impedance_by_sid(sid):
    """
    Get database/reference impedance for one simulation ID.

    Returns:
        mag_ohm   : |Z11| in Ohm, shape [F]
        phase_rad : phase in radians, shape [F]
        y         : raw ln_phase data, shape [2, F]
    """
    sid = int(sid)

    if sid not in sid_to_cache_pos:
        raise KeyError(f"SID {sid} not found in impedance cache.")

    pos = sid_to_cache_pos[sid]

    y = impedance_data[pos].cpu().numpy()  # [2, F]
    mag_ohm = np.exp(y[0])
    phase_rad = y[1]

    return mag_ohm, phase_rad, y


def load_z11_from_s4p(s4p_path):
    """
    Load Z11 from an EM .s4p file using scikit-rf.

    Returns:
        freq_hz   : frequency array [F]
        z11       : complex Z11 [F]
        mag_ohm   : |Z11| [F]
        phase_rad : phase(Z11) [F]
    """
    ntwk = rf.Network(s4p_path)

    freq_hz_em = ntwk.f
    z = ntwk.z
    z11 = z[:, 0, 0]

    mag_ohm = np.abs(z11)
    phase_rad = np.angle(z11)

    return freq_hz_em, z11, mag_ohm, phase_rad


def mae_ohm(a, b):
    return float(np.mean(np.abs(a - b)))


def mae_db(a, b):
    a_db = 20.0 * np.log10(a + 1e-12)
    b_db = 20.0 * np.log10(b + 1e-12)
    return float(np.mean(np.abs(a_db - b_db)))


def improvement_pct(baseline_error, optimized_error):
    return float(100.0 * (baseline_error - optimized_error) / baseline_error)


def parse_em_filename(s4p_path, regex):
    """
    Parse target_sid and start_sid from filename.
    Expected:
        target_100005_start_116459_geometry_impedance_loss_out.s4p
    """
    fname = os.path.basename(s4p_path)
    match = re.match(regex, fname)

    if match is None:
        raise ValueError(f"Filename does not match expected pattern:\n{fname}")

    target_sid = int(match.group(1))
    start_sid = int(match.group(2))

    return target_sid, start_sid

In [5]:
# Cell 5 - Discover NN EM .s4p files

s4p_files = sorted(glob.glob(os.path.join(EM_CFG["em_input_dir"], "*.s4p")))

if len(s4p_files) == 0:
    raise FileNotFoundError(
        f"No .s4p files found in:\n{EM_CFG['em_input_dir']}"
    )

EM_VALIDATION_FILES = []

for s4p_path in s4p_files:
    target_sid, start_sid = parse_em_filename(
        s4p_path=s4p_path,
        regex=EM_CFG["filename_regex"],
    )

    EM_VALIDATION_FILES.append({
        "design_type": "nn",
        "target_sid": target_sid,
        "start_sid": start_sid,
        "s4p_path": s4p_path,
        "s4p_file": os.path.basename(s4p_path),
    })

print("Found NN EM validation files:", len(EM_VALIDATION_FILES))

for item in EM_VALIDATION_FILES:
    print(
        f"Target {item['target_sid']} | "
        f"Start {item['start_sid']} | "
        f"{item['s4p_file']}"
    )

Found NN EM validation files: 10
Target 100005 | Start 116459 | target_100005_start_116459_geometry_impedance_loss_out.s4p
Target 100200 | Start 106556 | target_100200_start_106556_geometry_impedance_loss_out.s4p
Target 105000 | Start 112859 | target_105000_start_112859_geometry_impedance_loss_out.s4p
Target 106456 | Start 107018 | target_106456_start_107018_geometry_impedance_loss_out.s4p
Target 109000 | Start 109390 | target_109000_start_109390_geometry_impedance_loss_out.s4p
Target 109090 | Start 101651 | target_109090_start_101651_geometry_impedance_loss_out.s4p
Target 109276 | Start 103334 | target_109276_start_103334_geometry_impedance_loss_out.s4p
Target 111028 | Start 119268 | target_111028_start_119268_geometry_impedance_loss_out.s4p
Target 119000 | Start 105825 | target_119000_start_105825_geometry_impedance_loss_out.s4p
Target 119090 | Start 119457 | target_119090_start_119457_geometry_impedance_loss_out.s4p


In [6]:
# Cell 6 - Validate one EM result

def validate_one_em_result(item, save_plots=True):
    """
    Validate one EM-simulated inverse-designed geometry.

    Compares:
        target true impedance
        start true impedance
        EM simulated optimized geometry impedance
    """
    target_sid = int(item["target_sid"])
    start_sid = int(item["start_sid"])
    s4p_path = item["s4p_path"]

    target_mag, target_phase, _ = get_true_impedance_by_sid(target_sid)
    start_mag, start_phase, _ = get_true_impedance_by_sid(start_sid)

    em_freq_hz, em_z11, em_mag, em_phase = load_z11_from_s4p(s4p_path)
    em_freq_mhz = em_freq_hz / 1e6

    same_len = len(em_freq_hz) == len(freq_hz)
    same_grid = same_len and np.allclose(
        em_freq_hz,
        freq_hz,
        rtol=1e-6,
        atol=1e-3,
    )

    if not same_grid:
        print(f"Warning: frequency grid differs for target {target_sid}, start {start_sid}.")
        print("Metrics assume same frequency grid. Consider interpolation if needed.")

    nearest_true_mae_ohm = mae_ohm(target_mag, start_mag)
    nearest_true_mae_db = mae_db(target_mag, start_mag)

    em_optimized_mae_ohm = mae_ohm(target_mag, em_mag)
    em_optimized_mae_db = mae_db(target_mag, em_mag)

    improvement_ohm = improvement_pct(nearest_true_mae_ohm, em_optimized_mae_ohm)
    improvement_db = improvement_pct(nearest_true_mae_db, em_optimized_mae_db)

    result = {
        "design_type": "nn",
        "target_sid": target_sid,
        "nearest_start_sid": start_sid,
        "s4p_file": item["s4p_file"],
        "s4p_path": s4p_path,

        "nearest_true_mae_ohm": float(nearest_true_mae_ohm),
        "nearest_true_mae_db": float(nearest_true_mae_db),

        "em_optimized_mae_ohm": float(em_optimized_mae_ohm),
        "em_optimized_mae_db": float(em_optimized_mae_db),

        "em_improvement_over_nearest_ohm_pct": float(improvement_ohm),
        "em_improvement_over_nearest_db_pct": float(improvement_db),

        "same_frequency_grid": bool(same_grid),
        "n_freq_database": int(len(freq_hz)),
        "n_freq_em": int(len(em_freq_hz)),
    }

    run_dir = os.path.join(
        EM_CFG["em_results_dir"],
        f"target_{target_sid}_start_{start_sid}"
    )

    os.makedirs(run_dir, exist_ok=True)

    metrics_path = os.path.join(run_dir, "metrics_em_validation.json")

    with open(metrics_path, "w") as f:
        json.dump(result, f, indent=2)

    if save_plots:
        # Magnitude plot
        mag_path = os.path.join(run_dir, "em_validation_magnitude.png")

        plt.figure(figsize=(9, 6))
        plt.loglog(
            freq_mhz,
            target_mag,
            "k-",
            linewidth=2.2,
            label=f"Target true SID={target_sid}",
        )
        plt.loglog(
            freq_mhz,
            start_mag,
            "--",
            linewidth=1.7,
            label=f"NN start true SID={start_sid}",
        )
        plt.loglog(
            em_freq_mhz,
            em_mag,
            "-.",
            linewidth=2.0,
            label="EM simulated optimized geometry",
        )

        plt.xlabel("Frequency (MHz)")
        plt.ylabel("|Z11| (Ohm)")
        plt.title(
            f"EM Validation of NN Inverse-Designed Geometry\n"
            f"Target SID={target_sid}, Start SID={start_sid}"
        )
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(mag_path, dpi=EM_CFG["dpi"], bbox_inches="tight")
        plt.close()

        # Phase plot
        phase_path = os.path.join(run_dir, "em_validation_phase.png")

        plt.figure(figsize=(9, 5))
        plt.semilogx(
            freq_mhz,
            np.rad2deg(target_phase),
            "k-",
            linewidth=2.2,
            label=f"Target true SID={target_sid}",
        )
        plt.semilogx(
            freq_mhz,
            np.rad2deg(start_phase),
            "--",
            linewidth=1.7,
            label=f"NN start true SID={start_sid}",
        )
        plt.semilogx(
            em_freq_mhz,
            np.rad2deg(em_phase),
            "-.",
            linewidth=2.0,
            label="EM simulated optimized geometry",
        )

        plt.xlabel("Frequency (MHz)")
        plt.ylabel("Phase (degree)")
        plt.ylim(-185, 185)
        plt.yticks([-180, -90, 0, 90, 180])
        plt.title(
            f"EM Validation Phase Comparison - NN Start\n"
            f"Target SID={target_sid}, Start SID={start_sid}"
        )
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(phase_path, dpi=EM_CFG["dpi"], bbox_inches="tight")
        plt.close()

    print(
        f"Done target {target_sid}, start {start_sid}: "
        f"nearest={nearest_true_mae_db:.3f} dB, "
        f"EM={em_optimized_mae_db:.3f} dB, "
        f"improvement={improvement_db:.2f}%"
    )

    return result

In [7]:
# Cell 7 - Run EM validation for all NN files

all_em_results = []

for item in EM_VALIDATION_FILES:
    result = validate_one_em_result(
        item=item,
        save_plots=EM_CFG["save_plots"],
    )
    all_em_results.append(result)

em_results_df = pd.DataFrame(all_em_results)

display(em_results_df)

Done target 100005, start 116459: nearest=1.805 dB, EM=2.541 dB, improvement=-40.78%
Done target 100200, start 106556: nearest=3.226 dB, EM=1.562 dB, improvement=51.60%
Done target 105000, start 112859: nearest=2.694 dB, EM=0.869 dB, improvement=67.76%
Done target 106456, start 107018: nearest=3.639 dB, EM=2.266 dB, improvement=37.73%
Done target 109000, start 109390: nearest=1.872 dB, EM=1.568 dB, improvement=16.24%
Done target 109090, start 101651: nearest=1.386 dB, EM=1.929 dB, improvement=-39.15%
Done target 109276, start 103334: nearest=2.052 dB, EM=0.943 dB, improvement=54.03%
Done target 111028, start 119268: nearest=1.608 dB, EM=1.350 dB, improvement=16.09%
Done target 119000, start 105825: nearest=2.291 dB, EM=2.111 dB, improvement=7.88%
Done target 119090, start 119457: nearest=2.000 dB, EM=4.384 dB, improvement=-119.23%


,design_type,target_sid,nearest_start_sid,s4p_file,s4p_path,nearest_true_mae_ohm,nearest_true_mae_db,em_optimized_mae_ohm,em_optimized_mae_db,em_improvement_over_nearest_ohm_pct,em_improvement_over_nearest_db_pct,same_frequency_grid,n_freq_database,n_freq_em
0,nn,100005,116459,target_100005_start_116459_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.435658,1.804766,0.453266,2.540823,-4.041632,-40.784082,True,334,334
1,nn,100200,106556,target_100200_start_106556_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,2.188008,3.226029,0.750998,1.561556,65.676626,51.595093,True,334,334
2,nn,105000,112859,target_105000_start_112859_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,1.410948,2.693866,0.407933,0.868563,71.088039,67.757762,True,334,334
3,nn,106456,107018,target_106456_start_107018_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,1.367189,3.638502,0.856173,2.265583,37.377113,37.733089,True,334,334
4,nn,109000,109390,target_109000_start_109390_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.468516,1.872228,0.330555,1.568160,29.446341,16.240995,True,334,334
5,nn,109090,101651,target_109090_start_101651_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.554014,1.386217,0.783746,1.928970,-41.466773,-39.153547,True,334,334
6,nn,109276,103334,target_109276_start_103334_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,2.148845,2.052431,1.164467,0.943491,45.809609,54.030576,True,334,334
7,nn,111028,119268,target_111028_start_119268_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,1.184857,1.608354,0.952561,1.349650,19.605367,16.085019,True,334,334
8,nn,119000,105825,target_119000_start_105825_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.816379,2.290951,0.444645,2.110523,45.534483,7.875682,True,334,334
9,nn,119090,119457,target_119090_start_119457_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.273058,1.999854,0.365628,4.384225,-33.901179,-119.227245,True,334,334


In [8]:
# Cell 8 - Save NN EM validation summary

summary_csv_path = os.path.join(
    EM_CFG["em_results_dir"],
    "em_validation_summary_nn.csv"
)

em_results_df.to_csv(summary_csv_path, index=False)

print("Saved NN EM validation summary:")
print(summary_csv_path)

display(
    em_results_df[
        [
            "target_sid",
            "nearest_start_sid",
            "nearest_true_mae_ohm",
            "em_optimized_mae_ohm",
            "em_improvement_over_nearest_ohm_pct",
            "nearest_true_mae_db",
            "em_optimized_mae_db",
            "em_improvement_over_nearest_db_pct",
            "same_frequency_grid",
        ]
    ].sort_values("em_improvement_over_nearest_db_pct", ascending=False)
)

Saved NN EM validation summary:
/home/tekb/Master_Thesis_KB/CodesAndData/Results/EM_Validation_NN/em_validation_summary_nn.csv


,target_sid,nearest_start_sid,nearest_true_mae_ohm,em_optimized_mae_ohm,em_improvement_over_nearest_ohm_pct,nearest_true_mae_db,em_optimized_mae_db,em_improvement_over_nearest_db_pct,same_frequency_grid
2,105000,112859,1.410948,0.407933,71.088039,2.693866,0.868563,67.757762,True
6,109276,103334,2.148845,1.164467,45.809609,2.052431,0.943491,54.030576,True
1,100200,106556,2.188008,0.750998,65.676626,3.226029,1.561556,51.595093,True
3,106456,107018,1.367189,0.856173,37.377113,3.638502,2.265583,37.733089,True
4,109000,109390,0.468516,0.330555,29.446341,1.872228,1.568160,16.240995,True
7,111028,119268,1.184857,0.952561,19.605367,1.608354,1.349650,16.085019,True
8,119000,105825,0.816379,0.444645,45.534483,2.290951,2.110523,7.875682,True
5,109090,101651,0.554014,0.783746,-41.466773,1.386217,1.928970,-39.153547,True
0,100005,116459,0.435658,0.453266,-4.041632,1.804766,2.540823,-40.784082,True
9,119090,119457,0.273058,0.365628,-33.901179,1.999854,4.384225,-119.227245,True


In [9]:
# Cell 9 - Aggregate NN EM validation statistics

n_total = len(em_results_df)

n_success_ohm = int((em_results_df["em_improvement_over_nearest_ohm_pct"] > 0).sum())
n_success_db = int((em_results_df["em_improvement_over_nearest_db_pct"] > 0).sum())

avg_ohm_all = float(em_results_df["em_improvement_over_nearest_ohm_pct"].mean())
avg_db_all = float(em_results_df["em_improvement_over_nearest_db_pct"].mean())

positive_df = em_results_df[em_results_df["em_improvement_over_nearest_db_pct"] > 0]

avg_ohm_positive = float(positive_df["em_improvement_over_nearest_ohm_pct"].mean())
avg_db_positive = float(positive_df["em_improvement_over_nearest_db_pct"].mean())

print("NN EM validation aggregate statistics")
print("-------------------------------------")
print(f"Total cases                       : {n_total}")
print(f"Successful cases in Ohm MAE        : {n_success_ohm}/{n_total}")
print(f"Successful cases in dB MAE         : {n_success_db}/{n_total}")
print(f"Average Ohm improvement all cases  : {avg_ohm_all:.2f}%")
print(f"Average dB improvement all cases   : {avg_db_all:.2f}%")
print(f"Average Ohm improvement successes  : {avg_ohm_positive:.2f}%")
print(f"Average dB improvement successes   : {avg_db_positive:.2f}%")

NN EM validation aggregate statistics
-------------------------------------
Total cases                       : 10
Successful cases in Ohm MAE        : 7/10
Successful cases in dB MAE         : 7/10
Average Ohm improvement all cases  : 23.51%
Average dB improvement all cases   : 5.22%
Average Ohm improvement successes  : 44.93%
Average dB improvement successes   : 35.90%
